# 02 · BANKING77 SetFit pilot

Read the seed-42 manifests prepared in 01 and verify each text and label before training. Use validation to evaluate the 8-shot pilot.

In [1]:

import hashlib
import json
import random
import time
from importlib import metadata
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, load_dataset
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score

cwd = Path.cwd()
PROJECT = next((p for p in (cwd, *cwd.parents) if (p / 'results/protocol/protocol.json').is_file()), None)
if PROJECT is None:
    raise FileNotFoundError('Run 01 to prepare results/protocol, then start 02 from the project or notebooks directory.')
protocol_dir = PROJECT / 'results' / 'protocol'
config = json.loads((protocol_dir / 'protocol.json').read_text(encoding='utf-8'))
SEED = config['seed']
K = config['shots_per_class']
BACKBONE = 'sentence-transformers/all-MiniLM-L6-v2'
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
# 01 saved a row ID, label and normalized-text digest for each selected row.
raw = load_dataset('PolyAI/banking77')
label_names = dict(enumerate(raw['train'].features['label'].names))
source = {}
for name in ('train', 'test'):
    frame = raw[name].to_pandas()
    frame.index = pd.Index([f'banking77_{name}_{i}' for i in range(len(frame))], name='source_id')
    source[name] = frame


def read_split(part, source_name):
    manifest = pd.read_csv(protocol_dir / f'banking77_{part}.csv', dtype=str, keep_default_na=False)
    ids = manifest['source_id']
    if ids.duplicated().any() or not ids.isin(source[source_name].index).all():
        raise ValueError(f'Duplicate or unknown source_id in {part}')
    rows = source[source_name].loc[ids.tolist(), ['text', 'label']].reset_index()
    # split/join collapses whitespace and trims both ends, matching 01.
    keys = rows['text'].map(lambda text: ' '.join(text.split()).casefold())
    digests = keys.map(lambda key: hashlib.sha256(key.encode('utf-8')).hexdigest())
    matches = rows['label'].astype(str).eq(manifest['label']) & digests.eq(manifest['text_sha256'])
    if not matches.all():
        raise ValueError(f"Label or text hash mismatch in {part}: {ids[~matches].iloc[0]}")
    return rows, set(digests)


pool, pool_keys = read_split('pool', 'train')
bank_val, val_keys = read_split('validation', 'train')
_, test_keys = read_split('test', 'test')
train, _ = read_split('8shot', 'train')
if pool_keys & val_keys or pool_keys & test_keys or val_keys & test_keys:
    raise ValueError('Cross-split normalized text overlap')
if not train['source_id'].isin(pool['source_id']).all():
    raise ValueError('The 8-shot sample includes a row outside the training pool')
counts = train['label'].value_counts()
if K != 8 or len(label_names) != 77 or set(counts.index) != set(label_names) or not counts.eq(K).all():
    raise ValueError('Expected 77 intents with exactly 8 examples each')
fewshot_train = Dataset.from_pandas(train[['text','label']], preserve_index=False)
print(f'{len(train)} training examples, {len(label_names)} intents, {K} examples per intent, {len(bank_val)} validation examples.')
assert torch.cuda.is_available(), 'CUDA is required for the GPU pilot acceptance.'
gpu_sample = torch.randn(64, 64, device='cuda', requires_grad=True)
(gpu_sample @ gpu_sample).sum().backward()
torch.cuda.synchronize()
print('Interpreter:', __import__('sys').executable)
print('GPU forward/backward passed:', torch.cuda.get_device_name(0))
del gpu_sample


C:\APP\Anaconda\envs\py3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using the latest cached version of the dataset since PolyAI/banking77 couldn't be found on the Hugging Face Hub (offline mode is enabled).


Found the latest cached dataset configuration 'default' at C:\Users\l_len\Documents\ChatGPT\UNNC_NLP\notebook_audit_tmp\datasets\PolyAI___banking77\default\0.0.0\1846984189f2bc0ac51c6fc3a600fa9e1714e0c6 (last modified on Tue Nov 10 17:16:17 2026).


616 training examples, 77 intents, 8 examples per intent, 1000 validation examples.


Interpreter: C:\APP\Anaconda\envs\py3\python.exe
GPU forward/backward passed: NVIDIA GeForce RTX 5060 Laptop GPU


## Train SetFit

Training time includes trainer initialization and fitting. Each GPU memory measurement reports peak allocated memory during that phase.

In [2]:
from setfit import SetFitModel, TrainingArguments, Trainer
model = SetFitModel.from_pretrained(BACKBONE)
args = TrainingArguments(output_dir=str(PROJECT / 'results' / 'setfit_pilot'), batch_size=16, num_epochs=1, num_iterations=5, body_learning_rate=2e-5, seed=SEED, report_to='none', save_strategy='no')
if torch.cuda.is_available():
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
start = time.perf_counter()
trainer = Trainer(model=model, args=args, train_dataset=fewshot_train)
trainer.train()
if torch.cuda.is_available(): torch.cuda.synchronize()
train_seconds = time.perf_counter() - start
train_peak = torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else None
print('Training seconds:', round(train_seconds, 2), 'peak allocated MiB:', train_peak)

Could not reach the Hub (Cannot reach https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable.). Using cached commit hash for 'sentence-transformers/all-MiniLM-L6-v2'.


model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.


Map:   0%|          | 0/616 [00:00<?, ? examples/s]

Map: 100%|██████████| 616/616 [00:00<00:00, 22426.32 examples/s]

***** Running training *****


  Num unique pairs = 6160


  Batch size = 16


  Num epochs = 1


Step,Training Loss
1,0.139802
50,0.108518
100,0.095444
150,0.078593
200,0.079028
250,0.073075
300,0.074879
350,0.066734


Training seconds: 26.7 peak allocated MiB: 674.89404296875


## Validate and inspect errors

Measure prediction time on validation, calculate Accuracy and Macro-F1, and save the summary and misclassified examples in `results`.

In [3]:
if torch.cuda.is_available():
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
start = time.perf_counter()
pred = model.predict(bank_val.text.tolist())
if torch.cuda.is_available(): torch.cuda.synchronize()
predict_seconds = time.perf_counter() - start
predict_peak = torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else None
pred = pred.detach().cpu().tolist() if isinstance(pred, torch.Tensor) else list(pred)
assert len(pred) == len(bank_val)
accuracy = accuracy_score(bank_val.label, pred)
macro_f1 = f1_score(bank_val.label, pred, average='macro', zero_division=0)
result = bank_val[['source_id','text','label']].copy().reset_index(drop=True)
result['predicted_label'] = pred
errors = result[result.label != result.predicted_label].copy()
errors['true_intent'] = errors.label.map(label_names)
errors['predicted_intent'] = errors.predicted_label.map(label_names)
print(f'Validation accuracy: {accuracy:.4f}; macro-F1: {macro_f1:.4f}; errors: {len(errors)}')
print('Prediction seconds:', round(predict_seconds,2), 'peak allocated MiB:', predict_peak)
display(errors[['text','true_intent','predicted_intent']].head(10))
training_settings = {'batch_size': args.batch_size, 'num_epochs': args.num_epochs, 'num_iterations': args.num_iterations, 'body_learning_rate': args.body_learning_rate, 'max_seq_length': model.model_body.max_seq_length}
summary = {'training_settings': training_settings, 'gpu': torch.cuda.get_device_name(0), 'seed':SEED, 'shots':K, 'backbone':BACKBONE, 'train_seconds':train_seconds, 'prediction_seconds':predict_seconds, 'train_peak_allocated_mib':train_peak, 'prediction_peak_allocated_mib':predict_peak, 'validation_accuracy':accuracy, 'validation_macro_f1':macro_f1, 'versions':{p:metadata.version(p) for p in ['torch','setfit','sentence-transformers','transformers','datasets','scikit-learn']}}
(PROJECT/'results'/'metrics').mkdir(parents=True,exist_ok=True)
(PROJECT/'results'/'predictions').mkdir(parents=True,exist_ok=True)
(PROJECT/'results'/'metrics'/'banking77_setfit_pilot.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
errors.to_csv(PROJECT/'results'/'predictions'/'banking77_validation_errors.csv',index=False)

Validation accuracy: 0.7690; macro-F1: 0.7640; errors: 231
Prediction seconds: 0.42 peak allocated MiB: 366.095703125


,text,true_intent,predicted_intent
0,What are the policies of disposable cards?,disposable_card_limits,get_disposable_virtual_card
1,Why did I get charged one pound seemingly rand...,extra_charge_on_statement,transaction_charged_twice
5,"I got a card, how do I get it in the app?",card_linking,order_physical_card
6,The machine only paid me a hundred when I want...,wrong_amount_of_cash_received,extra_charge_on_statement
11,It is common for an international transfer to ...,balance_not_updated_after_bank_transfer,transfer_timing
18,why isn't my top-up going through?,pending_top_up,top_up_reverted
23,Please explain why my top-up hasn't been proce...,pending_top_up,top_up_reverted
24,How long does it take for a new card to ship?,card_arrival,card_delivery_estimate
26,Why is the transaction not posting?,pending_transfer,transfer_not_received_by_recipient
27,I am 16 and just started working at McDonalds ...,age_limit,top_up_by_cash_or_cheque
